In [1]:
import torch 
import torch.nn as nn


In [2]:
x = torch.randn(64, 32)
x1, x2 = torch.chunk(x, 2, dim=-1)


In [3]:
x.shape

torch.Size([64, 32])

In [4]:
import math
math.log(4)

1.3862943611198906

In [5]:
10**(-0.1)

0.7943282347242815

In [10]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        self.d_model = d_model
        self.num_heads = num_heads
        
        super(MultiHeadAttention, self).__init__()
        assert self.d_model%self.num_heads==0, "d_model must be divisible by num_heads"
        
        self.head_dim = self.d_model//self.num_heads
        
        self.query_proj = nn.Linear(d_model, d_model)
        self.key_proj = nn.Linear(d_model, d_model)
        self.value_proj = nn.Linear(d_model, d_model)
        
        self.out_proj = nn.Linear(d_model, d_model)
        self.attention = ScaledDotProductAttention()
    
    def split_heads(self, x):
        '''
        
        :param x: [batch_size, seq_len, d_model]
        :return: 
            [batch_size, num_heads, seq_len, head_dim]
        '''
        batch_size, seq_len, d_model = x.size()
        assert  d_model==self.head_dim*self.num_heads, f"input must in dim {self.num_heads*self.head_dim} but input dim is {d_model}"
        
        return x.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1,2)
    def combine_heads(self,x):
        '''
        
        :param x:  [batch_size, num_heads, seq_len, head_dim]
        :return:  [batch_size, seq_len, num_heads*head_dim]
        '''
        batch_size, num_heads, seq_len, head_dim = x.size()
        
        return x.transpose(1,2).contiguous().view(batch_size, seq_len,num_heads*head_dim)
        
    def forward(self, x, mask=None):
        
        query = self.query_proj(x)
        key = self.key_proj(x)
        value = self.value_proj(x)
        
        splited_query = self.split_heads(query)
        splited_key = self.split_heads(key)
        splited_value = self.split_heads(value)
        
        output, scores =self.attention(splited_query, splited_key, splited_value, mask) 
        output = self.combine_heads(output)
        
        return self.out_proj(output), scores

In [11]:
class MaskedMultiHeadAttention(MultiHeadAttention):
    """
    Masked Multi-Head Attention Module

    This layer is specifically used in the Transformer decoder’s self-attention block.
    It adds a future mask (causal mask) to prevent a position i from attending 
    to any future position j > i.
    """
    
    def __init__(self, d_model, num_heads):
        super(MaskedMultiHeadAttention, self).__init__(d_model, num_heads)
    
    def forward(self, x):
        """
        Compute masked multi-head self-attention.

        Args:
            x: Input tensor [batch_size, seq_len, d_model]

        Returns:
            output: Masked attention output [batch_size, seq_len, d_model]
            attention_weights: Attention weights 
                               [batch_size, num_heads, seq_len, seq_len]
        """
        seq_len = x.size(1)
        
        # Create a lower-triangular (causal) mask: 1s on and below the diagonal, 0s above
        # Shape: [1, seq_len, seq_len]
        mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0)
        
        # Call the parent class forward method with the mask applied
        return super().forward(x, mask)

In [13]:
batch_size, seq_len, d_model = 16, 20, 768

mmha = MaskedMultiHeadAttention(d_model, num_heads=12)
x = torch.randn(batch_size, seq_len, d_model)

output, scores = mmha(x)


print(f"output size is {output.size()}")
print(f"score size is {scores.size()}")
print(f"score is {scores[0]}")

output size is torch.Size([16, 20, 768])
score size is torch.Size([16, 12, 20, 20])
score is tensor([[[1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.3174, 0.6826, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.2499, 0.4439, 0.3062,  ..., 0.0000, 0.0000, 0.0000],
         ...,
         [0.0374, 0.0309, 0.0498,  ..., 0.0396, 0.0000, 0.0000],
         [0.0380, 0.0823, 0.0316,  ..., 0.0838, 0.0638, 0.0000],
         [0.0604, 0.0733, 0.0437,  ..., 0.0495, 0.0526, 0.0579]],

        [[1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.2656, 0.7344, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.3294, 0.4342, 0.2363,  ..., 0.0000, 0.0000, 0.0000],
         ...,
         [0.0657, 0.0387, 0.0686,  ..., 0.0455, 0.0000, 0.0000],
         [0.0451, 0.0512, 0.0433,  ..., 0.0352, 0.0578, 0.0000],
         [0.0402, 0.0652, 0.0851,  ..., 0.0454, 0.0508, 0.0633]],

        [[1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.6120, 0.3880, 0.00

In [19]:
class RotaryEmbedding(nn.Module):
    def __init__(self, base, head_dim, training_seq_len=2048):
        """
        Rotary Positional Embedding (RoPE) with NTK-aware scaling.
        
        Args:
            base (float): Base value for the rotary position encoding.
            head_dim (int): Dimensionality per attention head.
            training_seq_len (int): Sequence length used during training, 
                                    for NTK-aware scaling adjustment.
        """
        super().__init__()
        self.base = base
        self.dim = head_dim
        self.training_seq_len = training_seq_len

    def get_ntk_alpha(self, true_seq_len: int) -> float:
        """
        Compute the NTK-aware scaling factor alpha.

        Formula:
            ntk_alpha ≈ 2^(ceil(log2(true_seq_len / training_seq_len) + 1)) - 1
        Ensures that ntk_alpha >= 1.
        """
        ratio = max(true_seq_len / self.training_seq_len, 1e-6)
        context_value = math.log(ratio, 2) + 1
        ntk_alpha = 2 ** math.ceil(context_value) - 1
        ntk_alpha = max(ntk_alpha, 1)
        return ntk_alpha

    def get_mscaling(self, scale: float = 1.0) -> float:
        """
        Compute the m-scaling factor for attention scaling.

        Args:
            scale (float): Ratio of true_seq_len / training_seq_len.

        Returns:
            float: scaling multiplier.
        """
        if scale <= 1:
            return 1.0
        return 0.1 * math.log(scale) + 1.0

    def forward(self, max_seq_len: int):
        """
        Compute cosine and sine embeddings for rotary positional encoding.

        This function combines NTK-aware scaling and attention scaling.

        Example (when head_dim = 64):
            cos(theta_0), cos(theta_1), ..., cos(theta_63),
            cos(theta_0), cos(theta_1), ..., cos(theta_63)
            
            sin(theta_0), sin(theta_1), ..., sin(theta_63),
            sin(theta_0), sin(theta_1), ..., sin(theta_63)

        Here, `m` is the position index (0, 1, ..., max_seq_len-1).

        theta_i = 1 / base^(2i/d)
        The frequency vector = [1/base^0, 1/base^(2/d), ..., 1/base^(2*(d/2 - 1)/d)]

        Args:
            max_seq_len (int): Maximum sequence length for rotary encoding.

        Returns:
            Tuple[Tensor, Tensor]: (cosine matrix, sine matrix), both of shape [max_seq_len, head_dim].
        """
        # 1) Compute NTK-aware base scaling
        ntk_alpha = self.get_ntk_alpha(max_seq_len)
        base = self.base * ntk_alpha ** (self.dim / (self.dim - 2))

        # 2) Compute m-scaling factor
        mscale = self.get_mscaling(max_seq_len / self.training_seq_len)

        # 3) Compute inverse frequency vector
        inv_freq = 1.0 / (base ** (torch.arange(0, self.dim, 2, dtype=torch.float32) / self.dim))

        # 4) Position indices
        seq = torch.arange(max_seq_len, dtype=torch.float32)

        # 5) Outer product -> [max_seq_len, head_dim/2]
        emb = torch.outer(seq, inv_freq)

        # 6) Duplicate to [max_seq_len, head_dim]
        emb = torch.cat([emb, emb], dim=-1)

        return torch.cos(mscale * emb), torch.sin(mscale * emb)

In [20]:
# Example usage:
# Same usage as in MultiheadAttention
rope = RotaryEmbedding(1000, 768)  # training_seq_len defaults to 2048


In [21]:
res = rope(20)

In [22]:
res[0][0]

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 